### RAG Piplines - Data Injestion to Vector DB Pipeline


In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\allam\AppData\Local\Temp\ipykernel_25292\1573117932.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [ ]:
### Read all PDFS in library

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Interview Questions.pdf
  ✓ Loaded 4 pages

Processing: Rithvik_Allamaneni_Resume.pdf
  ✓ Loaded 1 pages

Total documents loaded: 5


[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-26T15:21:35-04:00', 'author': 'Rithvik Allamaneni', 'moddate': '2026-08-26T15:21:35-04:00', 'source': '..\\data\\Interview Questions.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Interview Questions.pdf', 'file_type': 'pdf'}, page_content='What to ask during interview \n \n- What kind of technologies will I work with \n- What kind of Training Will a I receive \n \n- Who I would work with on a daily basis and what would my schedule be like. \n \n \n- Any advice you would give to a new hire \no Curious Problem Solver \n- What would make you want to give a return offer to an intern. \no State their opinion and contribute to the conversation \no Be flexible and involve yourself \n \n- Is there anything I said that you might have concerns about? \n \n \nMolly and Jacob \n \n \nScenarios \n- Tell me about yourself \n- Is there a

In [14]:
### Text splitting into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs



In [17]:
chunks = split_documents(documents=all_pdf_documents)
chunks

Split 5 documents into 9 chunks

Example chunk:
Content: What to ask during interview 
 
- What kind of technologies will I work with 
- What kind of Training Will a I receive 
 
- Who I would work with on a daily basis and what would my schedule be like. 
...
Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-26T15:21:35-04:00', 'author': 'Rithvik Allamaneni', 'moddate': '2026-08-26T15:21:35-04:00', 'source': '..\\data\\Interview Questions.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Interview Questions.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-26T15:21:35-04:00', 'author': 'Rithvik Allamaneni', 'moddate': '2026-08-26T15:21:35-04:00', 'source': '..\\data\\Interview Questions.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Interview Questions.pdf', 'file_type': 'pdf'}, page_content='What to ask during interview \n \n- What kind of technologies will I work with \n- What kind of Training Will a I receive \n \n- Who I would work with on a daily basis and what would my schedule be like. \n \n \n- Any advice you would give to a new hire \no Curious Problem Solver \n- What would make you want to give a return offer to an intern. \no State their opinion and contribute to the conversation \no Be flexible and involve yourself \n \n- Is there anything I said that you might have concerns about? \n \n \nMolly and Jacob \n \n \nScenarios \n- Tell me about yourself \n- Is there a

In [18]:
### Embeddings and VectorStoreDB
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\allam\Documents\PythonProjects\LangChain_Intro\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
